# 🧠 Notebook 4: Deep Learning Pipeline (BONUS)

**Môn học:** Học Máy (CO3001) — Học kỳ I (2025-2026)  
**Nhóm:** 13

### Mục tiêu
- Huấn luyện MLP với PyTorch (nếu có)
- So sánh Deep Learning vs Traditional ML trên Adult dataset


In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import os, urllib.request, warnings, joblib, time

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report

warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (12, 5)

for d in ['../data','data','../features','features','../models','models',
          '../reports/figures','reports/figures','../reports','reports']:
    os.makedirs(d, exist_ok=True)

RANDOM_STATE = 42
DATA_DIR = '../data'    if os.path.isdir('../data')    else 'data'
FEAT_DIR = '../features' if os.path.isdir('../features') else 'features'
MOD_DIR  = '../models'  if os.path.isdir('../models')  else 'models'
REP_DIR  = '../reports' if os.path.isdir('../reports') else 'reports'
FIG_DIR  = os.path.join(REP_DIR, 'figures')

# Kiểm tra PyTorch
TORCH_AVAILABLE = False
try:
    import torch, torch.nn as nn, torch.optim as optim
    from torch.utils.data import TensorDataset, DataLoader
    TORCH_AVAILABLE = True
    print(f'✅ PyTorch: {torch.__version__}')
except ImportError:
    print('⚠️  PyTorch chưa cài → pip install torch')
    print('   Phần Bonus sẽ bị bỏ qua.')

# Kiểm tra TabNet
TABNET_AVAILABLE = False
try:
    from pytorch_tabnet.tab_model import TabNetClassifier
    TABNET_AVAILABLE = True
    print('✅ TabNet: OK')
except ImportError:
    print('⚠️  TabNet chưa cài → pip install pytorch-tabnet')


## 2. Load Data & Preprocessing

In [ ]:
COLUMNS    = ['age','workclass','fnlwgt','education','education_num',
              'marital_status','occupation','relationship','race','sex',
              'capital_gain','capital_loss','hours_per_week','native_country','income']
TRAIN_URL  = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
TEST_URL   = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.test"
TRAIN_PATH = os.path.join(DATA_DIR, 'adult_train.csv')
TEST_PATH  = os.path.join(DATA_DIR, 'adult_test.csv')

if not os.path.exists(TRAIN_PATH):
    urllib.request.urlretrieve(TRAIN_URL, TRAIN_PATH)
    urllib.request.urlretrieve(TEST_URL,  TEST_PATH)

df_train = pd.read_csv(TRAIN_PATH, names=COLUMNS, sep=r',\s*', engine='python', na_values='?')
df_test  = pd.read_csv(TEST_PATH,  names=COLUMNS, sep=r',\s*', engine='python', na_values='?', skiprows=1)
df_full  = pd.concat([df_train, df_test], ignore_index=True)
df_full['income'] = df_full['income'].str.replace('.', '', regex=False).str.strip()
df = df_full.drop_duplicates().reset_index(drop=True)

numeric_features     = ['age','fnlwgt','education_num','capital_gain','capital_loss','hours_per_week']
categorical_features = ['workclass','education','marital_status','occupation',
                        'relationship','race','sex','native_country']

X = df.drop(columns=['income'])
y = (df['income'] == '>50K').astype(int)

X_train_all, X_test, y_train_all, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

# Validation split from train
X_train, X_val, y_train, y_val = train_test_split(
    X_train_all, y_train_all, test_size=0.15, random_state=RANDOM_STATE, stratify=y_train_all)

print(f'Train: {X_train.shape}  |  Val: {X_val.shape}  |  Test: {X_test.shape}')

# Preprocessing
from sklearn.preprocessing import MinMaxScaler
preprocessor = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), numeric_features),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')),
                      ('enc', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), categorical_features),
])

X_train_proc = preprocessor.fit_transform(X_train, y_train).astype(np.float32)
X_val_proc   = preprocessor.transform(X_val).astype(np.float32)
X_test_proc  = preprocessor.transform(X_test).astype(np.float32)

print(f'Preprocessed: train={X_train_proc.shape}, val={X_val_proc.shape}, test={X_test_proc.shape}')


## 3. MLP Model (PyTorch)

In [ ]:
if TORCH_AVAILABLE:
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'Device: {device}')
    
    class MLP(nn.Module):
        def __init__(self, input_dim, hidden=[256, 128, 64], dropout=0.3):
            super().__init__()
            layers, prev = [], input_dim
            for h in hidden:
                layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
                prev = h
            layers.append(nn.Linear(prev, 2))
            self.net = nn.Sequential(*layers)
        def forward(self, x):
            return self.net(x)
    
    # Setup
    pos_w    = (y_train == 0).sum() / (y_train == 1).sum()
    X_tr_t   = torch.FloatTensor(X_train_proc).to(device)
    y_tr_t   = torch.LongTensor(y_train.values).to(device)
    X_val_t  = torch.FloatTensor(X_val_proc).to(device)
    y_val_t  = torch.LongTensor(y_val.values).to(device)
    X_te_t   = torch.FloatTensor(X_test_proc).to(device)
    
    dataset   = TensorDataset(X_tr_t, y_tr_t)
    loader    = DataLoader(dataset, batch_size=256, shuffle=True)
    criterion = nn.CrossEntropyLoss(weight=torch.FloatTensor([1.0, pos_w]).to(device))
    
    mlp_model = MLP(X_train_proc.shape[1]).to(device)
    optimizer = optim.Adam(mlp_model.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)
    
    print(f'MLP parameters: {sum(p.numel() for p in mlp_model.parameters()):,}')
    print('Starting training...')
else:
    print('⏭️  Bỏ qua — PyTorch chưa cài.')


In [ ]:
if TORCH_AVAILABLE:
    EPOCHS = 30
    history = {'train_loss': [], 'train_acc': [], 'val_acc': []}
    best_val_acc, patience_counter, best_state = 0, 0, None
    
    for epoch in range(EPOCHS):
        mlp_model.train()
        total_loss, correct, total = 0, 0, 0
        for bx, by in loader:
            optimizer.zero_grad()
            out  = mlp_model(bx)
            loss = criterion(out, by)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            correct    += out.argmax(1).eq(by).sum().item()
            total      += by.size(0)
        
        # Validation
        mlp_model.eval()
        with torch.no_grad():
            val_pred = mlp_model(X_val_t).argmax(1).cpu().numpy()
            val_acc  = accuracy_score(y_val, val_pred)
        
        train_acc = correct / total
        history['train_loss'].append(total_loss / len(loader))
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state   = {k: v.clone() for k, v in mlp_model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= 8:
                print(f'Early stopping tại epoch {epoch+1}')
                break
        
        if (epoch+1) % 5 == 0:
            print(f'Epoch {epoch+1:3d}/{EPOCHS} — loss={history["train_loss"][-1]:.4f}  '
                  f'train_acc={train_acc:.4f}  val_acc={val_acc:.4f}')
        
        scheduler.step()
    
    # Load best weights
    mlp_model.load_state_dict(best_state)
    print(f'\n✅ MLP training hoàn thành! Best val_acc: {best_val_acc:.4f}')


In [ ]:
if TORCH_AVAILABLE:
    # Training history plot
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    ax1.plot(history['train_loss'], 'b-o', markersize=4)
    ax1.set_title('Training Loss', fontsize=13, fontweight='bold')
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.grid(True, alpha=0.3)
    
    ax2.plot(history['train_acc'], 'b-o', markersize=4, label='Train')
    ax2.plot(history['val_acc'],   'r-s', markersize=4, label='Val')
    ax2.set_title('Accuracy', fontsize=13, fontweight='bold')
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy'); ax2.legend(); ax2.grid(True, alpha=0.3)
    
    plt.suptitle('MLP Training History', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{FIG_DIR}/mlp_training_history.png', dpi=150, bbox_inches='tight')
    plt.show()


In [ ]:
if TORCH_AVAILABLE:
    # Evaluate MLP on test set
    mlp_model.eval()
    with torch.no_grad():
        out         = mlp_model(X_te_t)
        y_pred_mlp  = out.argmax(1).cpu().numpy()
        y_proba_mlp = torch.softmax(out, 1)[:,1].cpu().numpy()
    
    mlp_acc = accuracy_score(y_test, y_pred_mlp)
    mlp_f1  = f1_score(y_test, y_pred_mlp)
    mlp_auc = roc_auc_score(y_test, y_proba_mlp)
    print(f'MLP  →  Accuracy: {mlp_acc:.4f}  F1: {mlp_f1:.4f}  AUC: {mlp_auc:.4f}')
    print()
    print(classification_report(y_test, y_pred_mlp, target_names=['<=50K', '>50K']))


## 4. TabNet Model

In [ ]:
if TABNET_AVAILABLE:
    tabnet = TabNetClassifier(
        n_d=16, n_a=16, n_steps=4, gamma=1.3,
        optimizer_params={'lr': 2e-2},
        scheduler_params={'step_size': 10, 'gamma': 0.9},
        max_epochs=50, patience=10, batch_size=1024,
        verbose=1
    )
    tabnet.fit(
        X_train_proc, y_train.values,
        eval_set=[(X_val_proc, y_val.values)],
        eval_metric=['accuracy']
    )
    y_pred_tabnet = tabnet.predict(X_test_proc)
    y_proba_tabnet = tabnet.predict_proba(X_test_proc)[:, 1]
    tn_acc = accuracy_score(y_test, y_pred_tabnet)
    tn_f1  = f1_score(y_test, y_pred_tabnet)
    tn_auc = roc_auc_score(y_test, y_proba_tabnet)
    print(f'\nTabNet →  Accuracy: {tn_acc:.4f}  F1: {tn_f1:.4f}  AUC: {tn_auc:.4f}')
else:
    print('⏭️  Bỏ qua TabNet — chưa cài pytorch-tabnet.')


## 5. So Sánh: Traditional ML vs Deep Learning

In [ ]:
# Load traditional ML results nếu có
trad_path = os.path.join(REP_DIR, 'model_results.csv')
if os.path.exists(trad_path):
    trad_df = pd.read_csv(trad_path, index_col=0)
    compare = trad_df[['accuracy','f1','roc_auc']].copy()
    compare['type'] = 'Traditional ML'
else:
    print('⚠️  Chưa có model_results.csv. Hãy chạy 03_Traditional_ML trước.')
    compare = pd.DataFrame(columns=['accuracy','f1','roc_auc','type'])

if TORCH_AVAILABLE:
    compare.loc['MLP'] = [mlp_acc, mlp_f1, mlp_auc, 'Deep Learning']
if TABNET_AVAILABLE:
    compare.loc['TabNet'] = [tn_acc, tn_f1, tn_auc, 'Deep Learning']

if not compare.empty:
    compare[['accuracy','f1','roc_auc']] = compare[['accuracy','f1','roc_auc']].astype(float)
    compare = compare.sort_values('f1', ascending=True)
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 7))
    for ax, metric in zip(axes, ['accuracy','f1','roc_auc']):
        colors = ['#e74c3c' if t=='Deep Learning' else '#3498db'
                  for t in compare['type']]
        bars = ax.barh(compare.index, compare[metric], color=colors, edgecolor='white', alpha=0.85)
        for bar, v in zip(bars, compare[metric]):
            ax.text(v+0.002, bar.get_y()+bar.get_height()/2,
                    f'{v:.4f}', va='center', fontsize=9)
        ax.set_xlabel(metric.upper())
        ax.set_title(metric.upper(), fontsize=12, fontweight='bold')
        ax.set_xlim(0.6, 1.0)
    
    from matplotlib.patches import Patch
    legend_elems = [Patch(facecolor='#3498db', label='Traditional ML'),
                    Patch(facecolor='#e74c3c', label='Deep Learning')]
    axes[0].legend(handles=legend_elems, fontsize=10)
    plt.suptitle('Traditional ML vs Deep Learning — Comparison', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{FIG_DIR}/ml_vs_dl_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print('\n=== Final Comparison ===')
    print(compare.sort_values('f1', ascending=False)[['accuracy','f1','roc_auc','type']].round(4).to_string())


## 6. Kết Luận

*(Điền nhận xét của nhóm)*

- **Model tốt nhất:** ...
- **Deep learning có vượt trội không?** ...
- **Lý do:** ...
- **Đề xuất cho production:** ...